# SCHISM external data and forcing

**Learning goals:** Prepare every available external forcing family—ERA5 atmosphere, HYCOM ocean fields, tidal harmonics, and wave spectra—and inspect the SCHISM files Rompy generates from them.

**Prerequisites:** Lesson 3; access to the shared `rom-py/rompy-test-data` fixture bundle.

**Execution contract:** This lesson is **configuration-only**. It runs Rompy data preparation but never executes SCHISM, downloads data during documentation rendering, or requires MPI/Docker.

## Checkpoint

For each source, record: source variables and coverage, Rompy object and coordinate mapping, generated file, and the structural/scientific checks still required.

Previous: [journey_03_schism_grid_data](../journey_03_schism_grid_data/)

Next: [journey_05_schism_boundaries](../journey_05_schism_boundaries/)


## Why this matters: one data-preparation workflow

**Without Rompy:** atmosphere, ocean boundaries, tidal harmonics, and wave spectra often arrive in different formats, coordinate systems, and time conventions. Each requires separate extraction, interpolation, naming, and model-file writing scripts.

**With Rompy:** source readers, variable mappings, filters, model grid, time range, and output conventions are composed into explicit objects. The cells below run those transformations and inspect their outputs. Rompy automates repetitive preparation; the modeller remains responsible for dataset choice, units, boundary conventions, and scientific validation.


In [ ]:
import sys
from pathlib import Path

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
             if (path / "scripts" / "schism_case_data.py").is_file())
sys.path.insert(0, str(root))
from scripts.schism_case_data import ensure_schism_data

case = ensure_schism_data()
print("Fixture directory:", case)


In [ ]:
import xarray as xr

# `case` is supplied by the previous lesson when running sequentially.
era5 = xr.open_dataset(case / "era5.nc")
hycom = xr.open_dataset(case / "hycom.nc")
print("ERA5 variables:", list(era5.data_vars))
print("HYCOM variables:", list(hycom.data_vars))
era5.close()
hycom.close()


## Visual verification: source fields

These are real cropped ERA5 and HYCOM fixtures. Plotting them before generation checks variable names, coordinate orientation, time coverage, and the spatial relationship to the regional mesh.


In [ ]:
import matplotlib.pyplot as plt
import xarray as xr

era5 = xr.open_dataset(case / "era5.nc")
hycom = xr.open_dataset(case / "hycom.nc")
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
era5.u10.isel(time=0).plot(ax=axes[0], cmap="coolwarm")
axes[0].set_title("ERA5 eastward wind")
hycom.surf_el.isel(time=0).plot(ax=axes[1], cmap="BrBG")
axes[1].set_title("HYCOM sea-surface elevation")
plt.show()
era5.close(); hycom.close()


## Atmospheric coverage and derived wind magnitude

ERA5 supplies more than one field. Here `u10` and `v10` are 10-m wind components and `msl` is mean sea-level pressure. Inspecting the extent and time axis before configuring Sflux catches reversed latitude axes, missing forecast hours, and unit assumptions early.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
era5 = xr.open_dataset(case / "era5.nc")
wind_speed = np.hypot(era5.u10, era5.v10)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
wind_speed.isel(time=0).plot(ax=axes[0], cmap="magma")
axes[0].quiver(era5.longitude, era5.latitude, era5.u10.isel(time=0), era5.v10.isel(time=0), color="white", scale=250)
axes[0].set_title("ERA5 10-m wind speed and vectors")
era5.msl.isel(time=0).plot(ax=axes[1], cmap="viridis")
axes[1].set_title("ERA5 mean sea-level pressure")
wind_speed.mean(dim=("latitude", "longitude")).plot(ax=axes[2])
axes[2].set_title("Wind time coverage")
axes[2].set_ylabel("speed (source units)")
plt.show()
print("ERA5 extent:", float(era5.longitude.min()), float(era5.longitude.max()), float(era5.latitude.min()), float(era5.latitude.max()))
print("ERA5 time range:", str(era5.time.min().values), "to", str(era5.time.max().values))
era5.close()


### Data-engineering and scientific checks

The source-to-Sflux transformation must preserve the intended time coverage and pressure/wind variables. Structural checks confirm dimensions and names only; they do not validate ERA5 provenance, unit conversion, land masking, or whether the atmospheric resolution is appropriate for this mesh.


## Rompy processing: ERA5 to SCHISM Sflux

The source plot above is only the input. Now Rompy performs the useful part: it selects the requested time window, applies the model-grid spatial context, maps `u10`, `v10`, and `msl` to SCHISM Sflux names, and writes a model-ready NetCDF file. The temporary directory keeps generated artefacts out of the repository.


In [ ]:
from tempfile import TemporaryDirectory
from rompy.core.data import DataBlob
from rompy.core.source import SourceFile
from rompy.core.filters import Filter
from rompy.core.time import TimeRange
from rompy_schism import SCHISMGrid
from rompy_schism.data import SCHISMDataSflux, SfluxAir

processing_grid = SCHISMGrid(
    hgrid=DataBlob(source=case / "hgrid.gr3"),
    vgrid=DataBlob(source=case / "vgrid.in"),
    drag=1,
)
atmos = SCHISMDataSflux(air_1=SfluxAir(
    source=SourceFile(uri=case / "era5.nc"),
    filter=Filter(sort={"coords": ["latitude"]}),
    uwind_name="u10", vwind_name="v10", prmsl_name="msl", buffer=2,
))
period = TimeRange(start="2023-01-01", end="2023-01-02", dt=3600)

with TemporaryDirectory() as output:
    generated = atmos.get(output, grid=processing_grid, time=period)
    sflux_file = next(Path(output).glob("sflux/air_*.nc"))
    sflux = xr.open_dataset(sflux_file)
    print("Rompy returned:", generated)
    print("Generated Sflux:", sflux_file.name)
    print("Generated dimensions:", dict(sflux.sizes))
    print("Generated variables:", list(sflux.data_vars))
    assert {"u10", "v10", "msl"}.issubset(sflux.data_vars)
    # Sflux is a regular atmospheric grid, so show the generated field over the
    # SCHISM mesh rather than reducing the result to a dimension-only chart.
    fig, ax = plt.subplots(figsize=(9, 5), constrained_layout=True)
    sflux.u10.isel(time=0).plot.pcolormesh(
        ax=ax, x="nx_grid", y="ny_grid", cmap="coolwarm", add_colorbar=True
    )
    processing_grid.plot(ax=ax)
    bx, by = processing_grid.boundary_points()
    ax.scatter(bx, by, s=4, c="yellow", edgecolor="black", label="SCHISM open boundary")
    ax.set_title("Rompy-generated Sflux wind over SCHISM mesh")
    ax.set_xlabel("longitude"); ax.set_ylabel("latitude"); ax.legend()
    plt.show()
    sflux.close()


## Full HYCOM variable family

The HYCOM fixture contains surface elevation plus depth-resolved `water_u`, `water_v`, `temperature`, and `salinity`. These are distinct contracts: elevation is 2-D in space, while the other variables require a depth/vertical-coordinate decision before writing SCHISM 3-D boundary files.


In [ ]:
hycom = xr.open_dataset(case / "hycom.nc")
fig, axes = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True)
for ax, name in zip(axes.flat, ["water_u", "water_v", "temperature", "salinity"]):
    hycom[name].isel(time=0, depth=0).plot(ax=ax, cmap="coolwarm")
    ax.set_title(f"HYCOM {name} at depth={float(hycom.depth.isel(depth=0)):.1f} m")
plt.show()
print("HYCOM dimensions:", dict(hycom.sizes))
print("3-D variables:", [name for name in ["water_u", "water_v", "temperature", "salinity"] if name in hycom])
hycom.close()


In [ ]:
tide_files = sorted((case / "tides" / "oceanum-atlas").glob("*.nc"))
print("Tidal constituents available:", sorted({p.name.split("_")[1].upper() for p in tide_files if "_" in p.name}))
print("Atlas files:", len(tide_files))
print("Configured assumptions: TPXO-style atlas, M2/S2/N2 constituents, bilinear interpolation, and open-boundary extraction.")
print("Nodal corrections and tidal potential must be chosen for the simulation epoch; their presence in configuration is not a validation of tidal skill.")


### Progressive boundary configuration

The existing elevation boundary is the supported, executable path. The 3-D variables above are intentionally inspected before enabling them: the SCHISM plugin must know the expected depth convention and generated file contract. Do not infer scientific validity from a successful interpolation alone.


## Rompy processing: HYCOM to an open-boundary file

The HYCOM plots show what Rompy receives. This cell now performs the transformation: Rompy uses the SCHISM mesh and coordinate mapping to interpolate `surf_el` onto the open-boundary nodes and writes `elev2D.th.nc`.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from rompy.core.data import DataBlob
from rompy.core.source import SourceFile
from rompy.core.time import TimeRange
from rompy_schism import SCHISMGrid
from rompy_schism.data import SCHISMDataBoundary

processing_grid = SCHISMGrid(
    hgrid=DataBlob(source=case / "hgrid.gr3"),
    vgrid=DataBlob(source=case / "vgrid.in"),
    drag=1,
)
elevation = SCHISMDataBoundary(
    id="elev2D", source=SourceFile(uri=case / "hycom.nc"),
    variables=["surf_el"], coords={"t": "time", "y": "ylat", "x": "xlon"},
)
period = TimeRange(start="2023-01-01", end="2023-01-02", dt=3600)
with TemporaryDirectory() as output:
    generated_file = Path(elevation.get(output, grid=processing_grid, time=period))
    generated = xr.open_dataset(generated_file)
    print("Rompy generated:", generated_file.name)
    print("Generated dimensions:", dict(generated.sizes))
    print("Generated variables:", list(generated.data_vars))
    assert "time_series" in generated.data_vars
    assert generated.sizes["nOpenBndNodes"] == len(processing_grid.boundary_points()[0])
    generated.time_series.isel(time=0, nLevels=0, nComponents=0).plot(color="steelblue")
    plt.title("Rompy-generated HYCOM elevation at open-boundary nodes")
    plt.xlabel("open-boundary node index")
    plt.show()
    generated.close()


In [ ]:
from rompy.core.data import DataBlob
from rompy_schism import SCHISMGrid
processing_grid = SCHISMGrid(
    hgrid=DataBlob(source=case / "hgrid.gr3"),
    vgrid=DataBlob(source=case / "vgrid.in"),
    drag=1,
)


## Tidal atlas data: from harmonic fields to `bctides.in`

Tidal forcing is not a time-series NetCDF like HYCOM elevation. Rompy reads harmonic atlas files: each constituent provides complex amplitude (`hRe`/`hIm`) on the atlas grid. It then evaluates the selected constituents for the run epoch, applies nodal corrections, samples them at the SCHISM open boundary, and writes `bctides.in`.


In [ ]:
import numpy as np

tide_file = case / "tides" / "oceanum-atlas" / "h_m2_tpxo9_atlas_30_v2.nc"
tide = xr.open_dataset(tide_file)
amplitude = np.hypot(tide.hRe, tide.hIm)
phase = np.degrees(np.arctan2(tide.hIm, tide.hRe))
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
mesh = axes[0].pcolormesh(tide.lon_z, tide.lat_z, amplitude.T, shading="auto", cmap="viridis")
axes[0].scatter(*processing_grid.boundary_points(), s=4, c="red")
axes[0].set_title("TPXO-style M2 amplitude")
axes[0].set_xlabel("longitude"); axes[0].set_ylabel("latitude")
fig.colorbar(mesh, ax=axes[0], label="atlas amplitude")
mesh = axes[1].pcolormesh(tide.lon_z, tide.lat_z, phase.T, shading="auto", cmap="twilight")
axes[1].set_title("TPXO-style M2 phase")
axes[1].set_xlabel("longitude"); axes[1].set_ylabel("latitude")
fig.colorbar(mesh, ax=axes[1], label="phase (degrees)")
plt.show()
print("Atlas variables:", list(tide.data_vars))
print("M2 complex field shape:", tide.hRe.shape)
tide.close()


In [ ]:
from tempfile import TemporaryDirectory
from rompy.core.time import TimeRange
from rompy_schism.boundary_conditions import create_tidal_only_boundary_config

tidal = create_tidal_only_boundary_config(
    constituents=["M2", "S2", "N2"],
    tidal_database=case / "tides",
    tidal_model="OCEANUM-atlas",
    nodal_corrections=True,
    tidal_potential=True,
    cutoff_depth=50.0,
)
with TemporaryDirectory() as output:
    result = tidal.get(output, grid=processing_grid, time=TimeRange(
        start="2023-01-01", end="2023-01-02", dt=3600
    ))
    bctides = Path(output) / "bctides.in"
    text = bctides.read_text()
    print("Rompy returned:", result)
    print("Generated tidal artefact:", bctides.name)
    print("Generated constituents:", [name for name in ["m2", "s2", "n2"] if name in text])
    print("Generated bctides preview:")
    print("\n".join(text.splitlines()[:18]))
    assert bctides.is_file()
    assert all(name in text for name in ["m2", "s2", "n2"])


## Wave spectra: from `ausspec` to SCHISM-WWM boundary input

The wave fixture is available through the local intake catalog. `SCHISMDataWave` selects spectrum stations close to the SCHISM boundary and serialises their spectra into the SCHISM-WWM NetCDF format. This is the wave analogue of the Sflux and HYCOM transformations above.


In [ ]:
from rompy.core.source import SourceIntake
from rompy_schism.data import SCHISMDataWave

waves = xr.open_dataset(root / "tests/data/aus-20230101.nc")

wave_source = SCHISMDataWave(
    id="wavedata",
    source=SourceIntake(dataset_id="ausspec", catalog_uri=root / "tests/data/catalog.yaml"),
    coords={"x": "lon", "y": "lat"},
    buffer=2.0,
)
with TemporaryDirectory() as output:
    wave_file = Path(wave_source.get(
        output, grid=processing_grid,
        time=TimeRange(start="2023-01-01", end="2023-01-02", dt=3600),
    ))
    wave_output = xr.open_dataset(wave_file)
    print("Rompy generated:", wave_file.name)
    print("Generated dimensions:", dict(wave_output.sizes))
    print("Generated variables:", list(wave_output.data_vars))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    axes[0].scatter(waves.lon, waves.lat, s=5, c="lightgray", label="source stations")
    axes[0].scatter(wave_output.longitude, wave_output.latitude, s=20, c="tab:orange", label="selected stations")
    axes[0].scatter(*processing_grid.boundary_points(), s=3, c="black", label="SCHISM boundary")
    axes[0].set_title("Rompy wave-station selection")
    axes[0].set_xlabel("longitude"); axes[0].set_ylabel("latitude"); axes[0].legend()
    spectrum = wave_output.efth.isel(time=0, station=0).sortby(["frequency", "direction"])
    spectrum.plot(ax=axes[1], x="frequency", y="direction", cmap="magma")
    axes[1].set_title("Generated WWM spectrum at selected station")
    plt.show()
    print("Wave hand-off: selected station spectra are now in SCHISM-WWM NetCDF format.")
    wave_output.close()
waves.close()
